## Live points quotes

Prices tonight's NBA points lines from the saved minutes and points-per-minute quantile models.

1. Read the latest points quotes from the Supabase `odds` schema, tables `nba_*`, and keep only games that tip on `SLATE_DATE`.
2. Build a pregame row for every quoted player and his team's assumed roster.
3. Predict the minutes quantiles and the points-per-minute quantiles.
4. Turn the rate knots into a mean rate and its coefficient of variation.
5. Build an integer points distribution from the exported Frank copula and NB2 count model in `copula_spec.json`, and compare it with each book's price.

The simulation is appearance-conditional. A blank in the inactive list is not priced as zero, and this notebook has no inactive list: the roster is every player whose latest silver appearance is with that team and within 45 days of the team's last game. A quoted player whose latest team is in the game is still added when that appearance is older. Minutes on the tonight row are that player's last positive minutes, so the starter grid treats him as available. Lagged minutes and rate features are shifted and do not read that fill.

`starting` on a tonight row is the RotoWire expected five for that NBA team. A player in the five is 1. A player off the five is 0. A team with no RotoWire game is not playing that date, so those players are skipped until their game is posted. History still uses the box-score `start_position`.

Season-to-date features, starter lookbacks, and days since the last appearance reset with the NBA season. On the first game of a season those features are empty. Silver through 2025-26 is history for the quote date. This notebook does not score that season. The copula and count model are the ones fit in `minutes_ppm_copula.ipynb` on the walk-forward OOS pairs. They are fixed, not refit to the quote date.


In [1]:
from hashlib import sha256
import json
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvinecopulib as pv
from IPython.display import Markdown, display
from scipy import stats
from sqlalchemy import text
from sqlalchemy.exc import SQLAlchemyError


def _repo_root() -> Path:
    starts = []
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        starts.append(Path(nb).resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / "pyproject.toml").is_file():
                return candidate
    raise RuntimeError("could not find repo root (pyproject.toml)")


ROOT = _repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from models.shared.artifacts import load_model_bundle, monotonize_quantiles, predict_quantiles
from models.shared.minutes_sampler import (
    QUANTILE_LEVELS,
    canonical_id,
    groups_for_frame as minutes_groups,
    load_tail_sidecar as load_minutes_tails,
    ppf as minutes_ppf,
)
from models.shared.points_given_minutes import MINUTES_CENTER, CountModel, conditional_points_pmf
from models.shared.ppm_sampler import (
    groups_for_frame as rate_groups,
    load_tail_sidecar as load_rate_tails,
    ppf as rate_ppf,
)
from models.shared.simulate import line_probs
from src.features.minutes import add_starter_features
from src.features.points.build import add_points_features
from src.scrapers.nba.nba_lineups import NBADailyLineups, posted_starting_flags
from src.features.points.pregame import OUTCOME_COLUMNS, appearances_before
from src.models.odds import american_to_implied_probability, expected_value, remove_vig
from src.pipeline.silver.positions import normalize_player_name, player_name_keys
from src.utils.db import get_engine

warnings.filterwarnings("ignore", category=DeprecationWarning, module="pyvinecopulib")

SEED = 42
KMAX = 90
MINUTES_GRID = 50
RATE_MEAN_GRID = 200
V_CLIP = 1e-12
LOOKBACK_HOURS = 36
ROSTER_WINDOW_DAYS = 45
# Eastern tip date to price. Set SLATE_DATE = "2026-10-02" to bet that day.
# Quotes whose game is not on this date are dropped.
SLATE_DATE = pd.Timestamp.now(tz="America/New_York").normalize().tz_localize(None)
POINTS_LABELS = {"points", "player_points", "pts"}
TEAM_BOOK_PRIORITY = {"novig": 0, "pinnacle": 1, "prophetx": 2}

PROP_TABLES = (
    ("novig", "nba_novig"),
    ("prophetx", "nba_prophetx"),
    ("fanduel", "nba_fanduel"),
    ("draftkings", "nba_draftkings"),
    ("pinnacle", "nba_pinnacle"),
    ("underdog", "nba_underdogs"),
    ("prizepicks", "nba_prizepicks"),
)
TEAM_TABLES = (
    ("novig", "nba_novig_team"),
    ("pinnacle", "nba_pinnacle_team"),
    ("prophetx", "nba_prophetx_team"),
)
ALLOWED_TABLES = {table for _, table in (*PROP_TABLES, *TEAM_TABLES)}

MINUTES_MODEL_PATH = ROOT / "models/saved_models/min_nba_model_2026-04-12.joblib"
RATE_MODEL_PATH = ROOT / "models/saved_models/pts_nba_model_2026-04-12.joblib"
MINUTES_TAILS_PATH = ROOT / "models/saved_models/min_nba_tails_2026-04-12.joblib"
RATE_TAILS_PATH = ROOT / "models/saved_models/pts_nba_tails_2026-04-12.joblib"
SPEC_PATH = ROOT / "artifacts/nba/copula/copula_spec.json"
SILVER_GLOB = "data/silver/nba/*/regular_season/player_gamelogs.parquet"

BLANK_COLUMNS = tuple(
    column for column in OUTCOME_COLUMNS if column not in {"minutes", "min", "min_sec"}
)
MINUTE_FILL_COLUMNS = ("minutes", "min", "min_sec")
MINUTES_KNOTS = [f"minutes_q_{level:.2f}" for level in QUANTILE_LEVELS]
RATE_KNOTS = [f"rate_q_{level:.2f}" for level in QUANTILE_LEVELS]
COUNT_FIELDS = ("log_kappa", "beta", "log_alpha", "beta2", "alpha_minutes", "alpha_cv", "gamma")
LINEAGE_FILES = {
    "minutes_bundle": MINUTES_MODEL_PATH,
    "ppm_bundle": RATE_MODEL_PATH,
    "minutes_tails": MINUTES_TAILS_PATH,
    "rate_tails": RATE_TAILS_PATH,
}


def nba_season_year(value) -> str:
    stamp = pd.Timestamp(value)
    start = stamp.year if stamp.month >= 10 else stamp.year - 1
    return f"{start}-{str(start + 1)[-2:]}"


def points_label(value) -> str:
    text_value = normalize_player_name(value).replace(" ", "_")
    return text_value


def is_points_market(value) -> bool:
    return points_label(value) in POINTS_LABELS


def market_labels(frame: pd.DataFrame) -> pd.Series:
    for column in ("stat_name", "market_type", "stat_type", "stat_category"):
        if column in frame.columns:
            return frame[column].map(points_label)
    return pd.Series("", index=frame.index)


def _stamp(frame: pd.DataFrame) -> pd.Series:
    fetched = frame["fetched_at"] if "fetched_at" in frame.columns else pd.Series(pd.NaT, index=frame.index)
    scraped = frame["scraped_at"] if "scraped_at" in frame.columns else pd.Series(pd.NaT, index=frame.index)
    return pd.to_datetime(fetched, utc=True, errors="coerce").fillna(
        pd.to_datetime(scraped, utc=True, errors="coerce")
    )


def read_recent(engine, table: str) -> pd.DataFrame:
    if table not in ALLOWED_TABLES:
        raise ValueError(f"unexpected odds table {table}")
    count = pd.read_sql(text(f"SELECT COUNT(*) AS n FROM odds.{table}"), engine)["n"].iloc[0]
    start, end = _slate_window()
    dated = text(
        f"""
        SELECT *
        FROM odds.{table}
        WHERE start_time >= :start
          AND start_time < :end
          AND COALESCE(fetched_at, scraped_at) >= (
              SELECT MAX(COALESCE(fetched_at, scraped_at))
              FROM odds.{table}
              WHERE start_time >= :start
                AND start_time < :end
          ) - make_interval(hours => :hours)
        """
    )
    lookback = text(
        f"""
        SELECT *
        FROM odds.{table}
        WHERE COALESCE(fetched_at, scraped_at) >= (
            SELECT MAX(COALESCE(fetched_at, scraped_at)) FROM odds.{table}
        ) - make_interval(hours => :hours)
        """
    )
    params = {
        "hours": LOOKBACK_HOURS,
        "start": start.to_pydatetime(),
        "end": end.to_pydatetime(),
    }
    scoped = True
    try:
        frame = pd.read_sql(dated, engine, params=params)
    except Exception as exc:
        if "start_time" not in str(exc):
            raise
        scoped = False
        frame = pd.read_sql(lookback, engine, params={"hours": LOOKBACK_HOURS})
    frame.attrs["table_rows"] = int(count)
    frame.attrs["slate_scoped"] = scoped
    if not frame.empty:
        frame["as_of"] = _stamp(frame)
    return frame


def _dedupe_latest(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    present = [column for column in columns if column in frame.columns]
    ordered = frame.sort_values("as_of")
    return ordered.drop_duplicates(present, keep="last")


def pair_sides(frame: pd.DataFrame, book: str) -> pd.DataFrame:
    work = frame.copy()
    work["book"] = book
    work["line_score"] = pd.to_numeric(work.get("line_score"), errors="coerce")
    work = work.loc[work["line_score"].notna()].copy()
    if work.empty:
        return work
    if "side" in work.columns:
        work["side"] = work["side"].astype("string").str.strip().str.lower()
    else:
        work["side"] = pd.NA
    for column in ("event_id", "home_team", "away_team", "player_name"):
        if column not in work.columns:
            work[column] = ""
        work[column] = work[column].fillna("").astype("string")
    if "start_time" not in work.columns:
        work["start_time"] = pd.NaT
    if "american_price" not in work.columns:
        work["american_price"] = np.nan
    work["american_price"] = pd.to_numeric(work["american_price"], errors="coerce")

    keys = ["book", "event_id", "player_name", "home_team", "away_team", "start_time", "line_score"]
    priced = work.loc[work["side"].isin(["over", "under"])].copy()
    if priced.empty:
        bare = _dedupe_latest(work, keys)
        bare["over_american"] = np.nan
        bare["under_american"] = np.nan
        return bare
    priced = _dedupe_latest(priced, keys + ["side"])
    wide = priced.pivot_table(
        index=keys,
        columns="side",
        values="american_price",
        aggfunc="last",
    ).reset_index()
    wide = wide.rename(columns={"over": "over_american", "under": "under_american"})
    for column in ("over_american", "under_american"):
        if column not in wide.columns:
            wide[column] = np.nan
    stamps = priced.groupby(keys, dropna=False)["as_of"].max().reset_index()
    return wide.merge(stamps, on=keys, how="left")


def load_points_quotes(engine) -> pd.DataFrame:
    """Latest points over/under from each nba_* prop table."""
    frames = []
    print(f"odds schema, lookback {LOOKBACK_HOURS}h from each table's newest stamp, games on {betting_slate().date()}")
    for book, table in PROP_TABLES:
        try:
            raw = read_recent(engine, table)
        except SQLAlchemyError as exc:
            print(f"{table}: {exc.__class__.__name__}")
            continue
        total = raw.attrs.get("table_rows", 0)
        if raw.empty:
            scope = "on this date" if raw.attrs.get("slate_scoped") else "in the latest window"
            print(f"{table}: {total} rows in the table, none {scope}")
            continue
        newest = raw["as_of"].max()
        age_hours = (pd.Timestamp.now(tz="UTC") - newest).total_seconds() / 3600
        labels = market_labels(raw)
        counts = labels.value_counts().head(8).to_dict()
        kept = raw.loc[labels.map(is_points_market)].copy()
        print(
            f"{table}: {total} rows, window {len(raw)}, points {len(kept)}, "
            f"newest {newest}, age {age_hours:.1f}h, markets {counts}"
        )
        if kept.empty:
            continue
        frames.append(pair_sides(kept, book))
    if not frames:
        return pd.DataFrame()
    quotes = pd.concat(frames, ignore_index=True)
    quotes = _fill_game_context(quotes)
    quotes["slate_date"] = _slate_series(quotes["start_time"])
    return _keep_slate(quotes, label="Paired points quotes")


def _fill_game_context(quotes: pd.DataFrame) -> pd.DataFrame:
    located = quotes.loc[
        quotes["home_team"].astype("string").str.len().gt(0)
        & quotes["away_team"].astype("string").str.len().gt(0)
    ]
    if located.empty:
        return quotes
    counts = located.groupby("player_name")[["home_team", "away_team"]].nunique()
    unique_players = counts.loc[(counts["home_team"] == 1) & (counts["away_team"] == 1)].index
    context = (
        located.loc[located["player_name"].isin(unique_players), ["player_name", "home_team", "away_team", "start_time"]]
        .drop_duplicates("player_name")
    )
    merged = quotes.merge(context, on="player_name", how="left", suffixes=("", "_ctx"))
    for column in ("home_team", "away_team", "start_time"):
        current = merged[column].replace("", pd.NA)
        merged[column] = current.fillna(merged[f"{column}_ctx"])
        merged = merged.drop(columns=f"{column}_ctx")
    return merged


def betting_slate() -> pd.Timestamp:
    """Eastern calendar date of the games to price."""
    stamp = pd.Timestamp(SLATE_DATE)
    if stamp.tzinfo is not None:
        stamp = stamp.tz_convert("America/New_York").tz_localize(None)
    return stamp.normalize()


def _slate_window() -> tuple[pd.Timestamp, pd.Timestamp]:
    start = betting_slate().tz_localize("America/New_York")
    return start, start + pd.Timedelta(days=1)


def _slate_series(start) -> pd.Series:
    when = pd.to_datetime(start, utc=True, errors="coerce")
    return when.dt.tz_convert("America/New_York").dt.normalize().dt.tz_localize(None)


def _keep_slate(frame: pd.DataFrame, *, label: str) -> pd.DataFrame:
    """Keep rows whose game tips on SLATE_DATE. Missing tip times are left out."""
    slate = betting_slate()
    if frame.empty:
        print(f"{label} on {slate.date()}: 0")
        return frame
    dates = frame["slate_date"] if "slate_date" in frame.columns else _slate_series(frame["start_time"])
    kept = frame.loc[dates.eq(slate)].copy()
    left_out = len(frame) - len(kept)
    print(f"{label} on {slate.date()}: {len(kept)}. Left out {left_out} from other dates or with no tip time.")
    return kept.reset_index(drop=True)


def load_team_lines(engine) -> pd.DataFrame:
    frames = []
    for book, table in TEAM_TABLES:
        try:
            raw = read_recent(engine, table)
        except SQLAlchemyError as exc:
            print(f"{table}: {exc.__class__.__name__}")
            continue
        total = raw.attrs.get("table_rows", 0)
        print(f"{table}: {total} rows, window {len(raw)}")
        if raw.empty or "market_type" not in raw.columns:
            continue
        work = raw.copy()
        work["book"] = book
        work["market_type"] = work["market_type"].astype("string").str.strip().str.lower()
        work["side"] = work["side"].astype("string").str.strip().str.lower() if "side" in work.columns else ""
        work["points"] = pd.to_numeric(work.get("points"), errors="coerce")
        work = work.loc[work["market_type"].isin(["spread", "total"]) & work["points"].notna()]
        if work.empty:
            continue
        if "start_time" not in work.columns:
            work["start_time"] = pd.NaT
        frames.append(
            work[
                [
                    "book",
                    "home_team",
                    "away_team",
                    "start_time",
                    "market_type",
                    "side",
                    "points",
                    "as_of",
                ]
            ]
        )
    if not frames:
        return pd.DataFrame(columns=["book", "home_team", "away_team", "market_type", "side", "points"])
    markets = pd.concat(frames, ignore_index=True)
    markets["slate_date"] = _slate_series(markets["start_time"])
    markets = _keep_slate(markets, label="Team lines")
    markets["priority"] = markets["book"].map(TEAM_BOOK_PRIORITY).fillna(9)
    return markets.sort_values("priority").reset_index(drop=True)


def load_silver() -> pd.DataFrame:
    paths = sorted(ROOT.glob(SILVER_GLOB))
    if not paths:
        raise FileNotFoundError(SILVER_GLOB)
    frames = [pd.read_parquet(path) for path in paths]
    silver = pd.concat(frames, ignore_index=True)
    silver["game_date"] = pd.to_datetime(silver["game_date"])
    silver["minutes"] = pd.to_numeric(silver["minutes"], errors="coerce")
    silver["starting"] = (
        silver["start_position"].astype("string").str.strip().isin(["G", "F", "C"]).astype(int)
    )
    return silver


def team_id_lookup(history: pd.DataFrame) -> dict[str, object]:
    latest = history.sort_values("game_date").groupby("team_id", as_index=False).tail(1)
    buckets: dict[str, set] = {}
    for row in latest.itertuples(index=False):
        keys = set()
        name = normalize_player_name(getattr(row, "team_name", ""))
        city = normalize_player_name(getattr(row, "team_city", ""))
        abbreviation = str(getattr(row, "team_abbreviation", "") or "").strip().lower()
        if name:
            keys.add(name)
            keys.add(name.split()[-1])
        if city and name:
            keys.add(normalize_player_name(f"{row.team_city} {row.team_name}"))
        if abbreviation:
            keys.add(abbreviation)
        for key in keys:
            buckets.setdefault(key, set()).add(row.team_id)
    return {key: next(iter(ids)) for key, ids in buckets.items() if len(ids) == 1}


def player_id_lookup(history: pd.DataFrame) -> dict[str, object]:
    ordered = history.sort_values(["game_date", "game_id"])
    lookup = {}
    for player_id, name in zip(ordered["player_id"], ordered["player_name"]):
        for key in player_name_keys(name):
            lookup[key] = player_id
    return lookup


def resolve_team(name, lookup):
    key = normalize_player_name(name)
    if not key:
        return None
    if key in lookup:
        return lookup[key]
    nickname = key.split()[-1]
    return lookup.get(nickname)


def resolve_player(name, lookup):
    for key in player_name_keys(name):
        if key in lookup:
            return lookup[key]
    return None


def _pick_line(markets: pd.DataFrame, home: str, away: str, market_type: str, side: str):
    if markets.empty:
        return np.nan
    home_key = normalize_player_name(home)
    away_key = normalize_player_name(away)
    rows = markets.loc[
        markets["home_team"].map(normalize_player_name).eq(home_key)
        & markets["away_team"].map(normalize_player_name).eq(away_key)
        & markets["market_type"].eq(market_type)
        & markets["side"].eq(side)
    ]
    if rows.empty:
        return np.nan
    return float(rows.iloc[0]["points"])


def game_environment(markets: pd.DataFrame, home: str, away: str) -> tuple[float, float, float]:
    home_spread = _pick_line(markets, home, away, "spread", "home")
    away_spread = _pick_line(markets, home, away, "spread", "away")
    if np.isnan(home_spread) and np.isfinite(away_spread):
        home_spread = -away_spread
    if np.isnan(away_spread) and np.isfinite(home_spread):
        away_spread = -home_spread
    total = _pick_line(markets, home, away, "total", "over")
    if np.isnan(total):
        total = _pick_line(markets, home, away, "total", "under")
    return home_spread, away_spread, total


def _matchup(abbreviation: str, opponent: str, is_home: bool) -> str:
    if is_home:
        return f"{abbreviation} vs. {opponent}"
    return f"{abbreviation} @ {opponent}"


def build_candidates(history: pd.DataFrame, quotes: pd.DataFrame, markets: pd.DataFrame, starters: dict, posted_teams: set):
    """Tonight rows for both teams, plus the quotes that matched a player and a game."""
    teams = team_id_lookup(history)
    players = player_id_lookup(history)
    latest_player = history.sort_values(["game_date", "game_id"]).groupby("player_id", as_index=False).tail(1)
    latest_player = latest_player.set_index("player_id", drop=False)
    positive = history.loc[history["minutes"].gt(0)].sort_values(["game_date", "game_id"])
    last_positive = positive.groupby("player_id", as_index=False).tail(1).set_index("player_id")
    team_meta = history.sort_values("game_date").groupby("team_id", as_index=False).tail(1).set_index("team_id")

    matched_rows = []
    unmatched_rows = []
    games: dict[tuple, dict] = {}
    for quote in quotes.itertuples(index=False):
        player_id = resolve_player(quote.player_name, players)
        home_id = resolve_team(quote.home_team, teams)
        away_id = resolve_team(quote.away_team, teams)
        if player_id is None:
            unmatched_rows.append({**quote._asdict(), "reason": "name not in silver"})
            continue
        if home_id is None or away_id is None:
            unmatched_rows.append({**quote._asdict(), "reason": "home or away team not in silver"})
            continue
        last = latest_player.loc[player_id]
        if last.team_id not in (home_id, away_id):
            unmatched_rows.append({**quote._asdict(), "reason": "player's last team is not in this game"})
            continue
        slate = pd.Timestamp(quote.slate_date)
        game_key = (slate, home_id, away_id)
        game_id = f"live-{canonical_id(away_id)}-{canonical_id(home_id)}-{slate:%Y%m%d}"
        games.setdefault(
            game_key,
            {
                "slate": slate,
                "home_id": home_id,
                "away_id": away_id,
                "home_name": quote.home_team,
                "away_name": quote.away_team,
                "game_id": game_id,
                "quoted_ids": set(),
            },
        )
        games[game_key]["quoted_ids"].add(player_id)
        matched_rows.append({**quote._asdict(), "player_id": player_id, "game_id": game_id})

    candidate_frames = []
    for game in games.values():
        home_spread, away_spread, total = game_environment(markets, game["home_name"], game["away_name"])
        for team_id, opponent_id, spread, is_home in (
            (game["home_id"], game["away_id"], home_spread, True),
            (game["away_id"], game["home_id"], away_spread, False),
        ):
            team_dates = history.loc[history["team_id"].eq(team_id), "game_date"]
            if team_dates.empty:
                continue
            cutoff = team_dates.max() - pd.Timedelta(days=ROSTER_WINDOW_DAYS)
            roster_ids = set(
                latest_player.loc[
                    latest_player["team_id"].eq(team_id) & latest_player["game_date"].ge(cutoff),
                    "player_id",
                ]
            )
            roster_ids.update(
                player_id
                for player_id in game["quoted_ids"]
                if latest_player.loc[player_id, "team_id"] == team_id
            )
            abbreviation = str(team_meta.loc[team_id, "team_abbreviation"])
            opponent_abbreviation = str(team_meta.loc[opponent_id, "team_abbreviation"])
            rows = []
            for player_id in roster_ids:
                template = latest_player.loc[player_id].copy()
                if player_id in last_positive.index:
                    filled = last_positive.loc[player_id]
                    for column in MINUTE_FILL_COLUMNS:
                        if column in template.index:
                            template[column] = filled["minutes"]
                for column in BLANK_COLUMNS:
                    if column in template.index:
                        template[column] = np.nan if column != "start_position" else ""
                if "comment" in template.index:
                    template["comment"] = ""
                template["game_date"] = game["slate"]
                template["game_id"] = game["game_id"]
                template["season_year"] = nba_season_year(game["slate"])
                template["team_id"] = team_id
                template["opp_team_id"] = opponent_id
                template["matchup"] = _matchup(abbreviation, opponent_abbreviation, is_home)
                template["player_team_spread"] = spread
                template["game_total"] = total
                template["is_quoted"] = player_id in game["quoted_ids"]
                rows.append(template)
            if rows:
                candidate_frames.append(pd.DataFrame(rows))

    candidates = pd.concat(candidate_frames, ignore_index=True) if candidate_frames else history.iloc[0:0].copy()
    if not candidates.empty:
        candidates = candidates.sort_values("is_quoted", ascending=False).drop_duplicates(
            ["game_id", "player_id"], keep="first"
        )
        candidates["starting"] = posted_starting_flags(
            candidates["player_name"],
            candidates["team_abbreviation"],
            starters,
            posted_teams,
        )
        playing = candidates["starting"].notna()
        skipped = candidates.loc[~playing, ["player_id", "game_id"]]
        candidates = candidates.loc[playing].copy()
        candidates["starting"] = candidates["starting"].astype(int)
    else:
        skipped = candidates.iloc[0:0]
    matched = pd.DataFrame(matched_rows)
    if not matched.empty and not skipped.empty:
        skipped_keys = set(zip(skipped["player_id"], skipped["game_id"]))
        has_game = pd.Series(
            [
                (player_id, game_id) not in skipped_keys
                for player_id, game_id in zip(matched["player_id"], matched["game_id"])
            ],
            index=matched.index,
        )
        held = matched.loc[~has_game].copy()
        held["reason"] = "no posted game"
        matched = matched.loc[has_game].reset_index(drop=True)
        unmatched_rows.extend(held.to_dict("records"))
    unmatched = pd.DataFrame(unmatched_rows)
    return candidates, matched, unmatched


def feature_panel(history: pd.DataFrame, candidates: pd.DataFrame) -> pd.DataFrame:
    panel = pd.concat([history, candidates], ignore_index=True)
    featured = add_starter_features(add_points_features(panel))
    game_ids = set(candidates["game_id"])
    return featured.loc[featured["game_id"].isin(game_ids)].copy()


def _bundle_status(bundle: dict) -> str:
    metadata = bundle.get("metadata") or {}
    if isinstance(metadata, dict) and metadata.get("status"):
        return str(metadata["status"])
    return "unlabeled"


def attach_quantiles(frame: pd.DataFrame, preds: pd.DataFrame, prefix: str) -> pd.DataFrame:
    out = frame.copy()
    for level in QUANTILE_LEVELS:
        source = f"q_{level:.2f}"
        out[f"{prefix}_q_{level:.2f}"] = preds[source].to_numpy()
    return out


def predict_knots(live: pd.DataFrame, minutes_bundle: dict, rate_bundle: dict) -> pd.DataFrame:
    quoted = live.loc[live["is_quoted"].astype(bool)].reset_index(drop=True)
    if quoted.empty:
        raise ValueError("no quoted players reached the feature panel")
    for bundle in (minutes_bundle, rate_bundle):
        missing = [name for name in bundle["feature_names"] if name not in quoted.columns]
        if missing:
            raise KeyError(f"missing model features {missing}")
    if quoted["starting"].isna().any():
        raise ValueError("starting is missing on a quoted row")
    quoted["starting"] = quoted["starting"].astype(int)

    minutes_x = quoted.loc[:, minutes_bundle["feature_names"]]
    minutes_raw = predict_quantiles(minutes_bundle["quantile_models"], minutes_bundle["feature_names"], minutes_x)
    minutes_mono = pd.DataFrame(
        monotonize_quantiles({column: minutes_raw[column].to_numpy() for column in minutes_raw.columns})
    )
    rate_x = quoted.loc[:, rate_bundle["feature_names"]]
    rate_raw = predict_quantiles(rate_bundle["quantile_models"], rate_bundle["feature_names"], rate_x)
    rate_mono = pd.DataFrame(
        monotonize_quantiles({column: rate_raw[column].to_numpy() for column in rate_raw.columns})
    )
    quoted = attach_quantiles(quoted, minutes_mono, "minutes")
    quoted = attach_quantiles(quoted, rate_mono, "rate")
    return quoted


def missing_feature_report(frame: pd.DataFrame, minutes_names, rate_names) -> pd.DataFrame:
    rows = []
    for model, names in (("minutes", minutes_names), ("rate", rate_names)):
        for name in names:
            rows.append({"model": model, "feature": name, "missing": float(frame[name].isna().mean())})
    report = pd.DataFrame(rows)
    return report.loc[report["missing"].gt(0)].reset_index(drop=True)


def file_sha256(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def load_simulator(path: Path = SPEC_PATH) -> dict:
    """Exported copula and NB2 count model from `copula_spec.json`."""
    spec = json.loads(path.read_text())
    block = spec["count_model"]
    bad = [name for name in COUNT_FIELDS if isinstance(block.get(name), str) or block.get(name) is None]
    if bad:
        raise ValueError(f"{path.name} has no numeric {bad}. Rerun the export cell in minutes_ppm_copula.ipynb.")
    if float(block["minutes_center"]) != MINUTES_CENTER:
        raise ValueError(f"spec minutes_center {block['minutes_center']} vs code {MINUTES_CENTER}")
    model = CountModel(**{name: float(block[name]) for name in COUNT_FIELDS})
    direct = not bool(spec["copula"]["used_in_simulation"])
    bicop = None if direct else pv.Bicop.from_json(spec["copula"]["pyvinecopulib_json"])
    stale = []
    for key, local in LINEAGE_FILES.items():
        stored = spec["lineage"]["files"].get(key, {})
        if stored.get("sha256") != file_sha256(local):
            stale.append(key)
    if direct:
        label = f"direct_nb2_gamma_{model.gamma:.3f}"
    else:
        label = f"{spec['copula']['family']}_{float(spec['copula']['parameters'][0]):.3f}_nb2"
    return {
        "spec": spec,
        "model": model,
        "bicop": bicop,
        "use_surprise": direct,
        "label": label,
        "stale": stale,
    }


def rate_moments(frame: pd.DataFrame, rate_tables) -> tuple[np.ndarray, np.ndarray]:
    """Mean rate and std / mean from a midpoint quadrature of the rate sampler."""
    grid_u = (np.arange(RATE_MEAN_GRID) + 0.5) / RATE_MEAN_GRID
    lower, upper = rate_groups(frame["starting"], rate_tables)
    draws = rate_ppf(
        np.tile(grid_u, (len(frame), 1)),
        frame[RATE_KNOTS].to_numpy(dtype=float),
        lower,
        upper,
        rate_tables,
    )
    mean = draws.mean(axis=1)
    return mean, draws.std(axis=1) / np.maximum(mean, 1e-9)


def points_pmfs(sim: pd.DataFrame, simulator: dict, minutes_tables) -> np.ndarray:
    """Points PMF per row, averaging the copula conditional over a minutes PIT grid."""
    u_grid = (np.arange(MINUTES_GRID) + 0.5) / MINUTES_GRID
    lower, upper = minutes_groups(sim["starting"], minutes_tables)
    minutes_at_u = minutes_ppf(
        np.tile(u_grid, (len(sim), 1)),
        sim[MINUTES_KNOTS].to_numpy(dtype=float),
        lower,
        upper,
        minutes_tables,
    )
    return conditional_points_pmf(
        minutes_at_u,
        u_grid,
        sim["rate_hat"].to_numpy(dtype=float),
        simulator["model"],
        kmax=KMAX,
        rate_cv=sim["rate_cv"].to_numpy(dtype=float),
        bicop=simulator["bicop"],
        use_surprise=simulator["use_surprise"],
    )


def nb_ppf(v, mu, alpha) -> np.ndarray:
    size = 1.0 / np.asarray(alpha, dtype=float)
    prob = size / (size + np.asarray(mu, dtype=float))
    q = np.clip(np.asarray(v, dtype=float), V_CLIP, 1.0 - V_CLIP)
    return stats.nbinom.ppf(q, size, prob)


def _american(value):
    if value is None or pd.isna(value):
        return np.nan
    number = float(value)
    if not np.isfinite(number) or number == 0:
        return np.nan
    return number


def _model_side(ev_over, ev_under) -> str:
    over_ok = pd.notna(ev_over) and ev_over > 0
    under_ok = pd.notna(ev_under) and ev_under > 0
    if over_ok and (not under_ok or ev_over >= ev_under):
        return "over"
    if under_ok:
        return "under"
    if pd.isna(ev_over) and pd.isna(ev_under):
        return ""
    return "pass"


def price_quotes(quoted: pd.DataFrame, matched: pd.DataFrame, simulator: dict, minutes_tables, rate_tables):
    sim = quoted.drop_duplicates(["player_id", "game_id"]).reset_index(drop=True)
    sim["rate_hat"], sim["rate_cv"] = rate_moments(sim, rate_tables)
    pmf = points_pmfs(sim, simulator, minutes_tables)
    sim["pmf_mean"] = pmf @ np.arange(pmf.shape[1], dtype=float)
    sim["pmf_median"] = np.argmax(np.cumsum(pmf, axis=1) >= 0.5, axis=1)
    lookup = {(row.player_id, row.game_id): index for index, row in enumerate(sim.itertuples(index=False))}
    rows = []
    for quote in matched.itertuples(index=False):
        index = lookup[(quote.player_id, quote.game_id)]
        over, push, under = line_probs(pmf[index], float(quote.line_score))
        over_american = _american(quote.over_american)
        under_american = _american(quote.under_american)
        raw_over = american_to_implied_probability(over_american) if pd.notna(over_american) else np.nan
        raw_under = american_to_implied_probability(under_american) if pd.notna(under_american) else np.nan
        if pd.notna(over_american) and pd.notna(under_american):
            no_vig_over, no_vig_under = remove_vig(over_american, under_american)
        else:
            no_vig_over, no_vig_under = np.nan, np.nan
        ev_over = expected_value(over, under, over_american) if pd.notna(over_american) else np.nan
        ev_under = expected_value(under, over, under_american) if pd.notna(under_american) else np.nan
        player = sim.loc[index]
        rows.append(
            {
                "book": quote.book,
                "player_name": quote.player_name,
                "team_name": player.team_name,
                "matchup": player.matchup,
                "line_score": float(quote.line_score),
                "starting": int(player.starting),
                "minutes_q50": float(player["minutes_q_0.50"]),
                "rate_q50": float(player["rate_q_0.50"]),
                "rate_hat": float(player.rate_hat),
                "rate_cv": float(player.rate_cv),
                "pmf_mean": float(player.pmf_mean),
                "pmf_median": int(player.pmf_median),
                "over_american": over_american,
                "under_american": under_american,
                "sim_over": float(over),
                "sim_under": float(under),
                "sim_push": float(push),
                "raw_over": raw_over,
                "raw_under": raw_under,
                "no_vig_over": no_vig_over,
                "no_vig_under": no_vig_under,
                "ev_over": ev_over,
                "ev_under": ev_under,
                "model_side": _model_side(ev_over, ev_under),
                "player_id": quote.player_id,
                "game_id": quote.game_id,
                "seed": SEED,
                "minutes_grid": MINUTES_GRID,
                "copula": simulator["label"],
            }
        )
    card = pd.DataFrame(rows)
    card["best_ev"] = card[["ev_over", "ev_under"]].max(axis=1)
    card = card.sort_values(["best_ev", "book", "player_name"], ascending=[False, True, True]).reset_index(drop=True)
    pmf_table = {key: pmf[index] for key, index in lookup.items()}
    return card, sim, pmf_table


def illustration_draws(player: pd.Series, simulator: dict, minutes_tables, line: float, n: int = 6) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    u = rng.random(n)
    w = rng.random(n)
    if simulator["bicop"] is None:
        v = w
    else:
        v = np.asarray(simulator["bicop"].hinv1(np.column_stack([u, w])), dtype=float)
    minute_grid = player[MINUTES_KNOTS].to_numpy(dtype=float).reshape(1, -1)
    lower, upper = minutes_groups(pd.Series([int(player["starting"])]), minutes_tables)
    minutes = minutes_ppf(u.reshape(1, -1), minute_grid, lower, upper, minutes_tables)[0]
    minutes = np.maximum(minutes, 1e-6)
    model = simulator["model"]
    surprise = stats.norm.ppf(u) if simulator["use_surprise"] else None
    mu = model.mean(minutes, float(player["rate_hat"]), surprise)
    alpha = model.alpha(minutes, float(player["rate_cv"]))
    points = nb_ppf(v, mu, alpha).astype(int)
    return pd.DataFrame(
        {
            "u_minutes": u,
            "v_points": v,
            "minutes": minutes,
            "nb2_mean": mu,
            "nb2_alpha": alpha,
            "points": points,
            "over": points > line,
        }
    )


def run_slate(quotes: pd.DataFrame, markets: pd.DataFrame) -> dict:
    slate = betting_slate()
    work = quotes.dropna(subset=["slate_date"]).copy()
    if work.empty:
        raise ValueError("quotes have no slate date")
    other_dates = int(work["slate_date"].ne(slate).sum())
    if other_dates:
        print(f"Pricing {slate.date()} and leaving {other_dates} quotes from other dates.")
    work = work.loc[work["slate_date"].eq(slate)].reset_index(drop=True)
    if work.empty:
        raise ValueError(f"no quotes for games on {slate.date()}")

    print(f"Loading silver through {pd.Timestamp(slate).date()}.")
    silver = load_silver()
    history = appearances_before(silver, slate)
    print("Loading NBA starting lineups from RotoWire.")
    lineups = NBADailyLineups()
    starters = lineups.expected_starters_by_abbr()
    posted = lineups.posted_teams()
    print(f"RotoWire posted games for {len(posted)} teams.")
    candidates, matched, unmatched = build_candidates(history, work, markets, starters, posted)
    skipped_quotes = 0
    if not unmatched.empty and "reason" in unmatched.columns:
        skipped_quotes = int(unmatched["reason"].eq("no posted game").sum())
    if skipped_quotes:
        print(f"Skipped {skipped_quotes} quotes whose team has no RotoWire game on this date.")
    print(
        f"Matched {matched['player_name'].nunique() if not matched.empty else 0} players "
        f"across {candidates['game_id'].nunique() if not candidates.empty else 0} games, "
        f"{len(candidates)} roster rows, {len(unmatched)} unmatched quotes."
    )
    if matched.empty:
        return {"card": matched, "unmatched": unmatched, "missing": pd.DataFrame(), "example": None}

    print("Building pregame features and predicting quantiles.")
    live = feature_panel(history, candidates)
    minutes_bundle = load_model_bundle(MINUTES_MODEL_PATH)
    rate_bundle = load_model_bundle(RATE_MODEL_PATH)
    print(f"minutes bundle status {_bundle_status(minutes_bundle)}, train end {minutes_bundle.get('train_end')}")
    print(f"rate bundle status {_bundle_status(rate_bundle)}, train end {rate_bundle.get('train_end')}")
    quoted = predict_knots(live, minutes_bundle, rate_bundle)
    missing = missing_feature_report(quoted, minutes_bundle["feature_names"], rate_bundle["feature_names"])

    simulator = load_simulator()
    spec = simulator["spec"]
    print(
        f"Pricing with {simulator['label']} from {SPEC_PATH.name}, status {spec.get('status')}, "
        f"{MINUTES_GRID}-point minutes grid."
    )
    if spec.get("status_note"):
        print(f"  spec note: {spec['status_note']}")
    if simulator["stale"]:
        print(f"  {', '.join(simulator['stale'])} changed since the copula was fit. Refit the copula on this bundle's OOS pairs.")
    minutes_tables = load_minutes_tails(MINUTES_TAILS_PATH)
    rate_tables = load_rate_tails(RATE_TAILS_PATH)
    card, sim, pmf_table = price_quotes(quoted, matched, simulator, minutes_tables, rate_tables)

    example_row = card.iloc[0]
    player = sim.loc[(sim["player_id"] == example_row.player_id) & (sim["game_id"] == example_row.game_id)].iloc[0]
    draws = illustration_draws(player, simulator, minutes_tables, float(example_row.line_score))
    knot_rows = []
    for prefix in ("minutes", "rate"):
        knot_rows.append(
            {
                "grid": prefix,
                **{f"{level:.2f}": float(player[f"{prefix}_q_{level:.2f}"]) for level in QUANTILE_LEVELS},
            }
        )
    return {
        "card": card,
        "unmatched": unmatched,
        "missing": missing,
        "example": {
            "quote": example_row,
            "player": player,
            "draws": draws,
            "knots": pd.DataFrame(knot_rows),
            "pmf": pmf_table[(example_row.player_id, example_row.game_id)],
        },
    }


Matplotlib is building the font cache; this may take a moment.


## Odds

Each prop table contributes the latest rows for games that tip on `SLATE_DATE` in Eastern time. Tables that store `start_time` are queried for that date only. Tables without a tip time still use the 36 hours before their newest stamp, then a quote is kept only when another book places that player's game on `SLATE_DATE`. Everyone else is left out. Points markets are `points`, `player_points`, and `pts`. Combo markets stay out. Over and under are paired on book, player, line, and game. PrizePicks has no American price, so those rows still get a simulated probability and a blank EV.

Team spread and total come from `nba_novig_team`, `nba_pinnacle_team`, and `nba_prophetx_team`, in that order when more than one book has the game. The home spread is the home team's `player_team_spread`.


In [2]:
engine = get_engine()
quotes = load_points_quotes(engine)
team_lines = load_team_lines(engine)
if quotes.empty:
    print("No points quotes in the latest window. Load the nba_* prop tables, then rerun from this cell.")
else:
    print(f"{len(quotes)} paired points quotes, {quotes['slate_date'].nunique()} slate date(s).")
    display(quotes.head(20))
if team_lines.empty:
    print("No spread or total in the team tables. implied_team_total and team_spread_canonical will be empty.")
else:
    display(team_lines.groupby(["book", "market_type"]).size().rename("rows").reset_index())


odds schema, lookback 36h from each table's newest stamp


nba_novig: 0 rows in the table, none in the latest window


nba_prophetx: 0 rows in the table, none in the latest window


nba_fanduel: 0 rows in the table, none in the latest window


nba_draftkings: 0 rows in the table, none in the latest window


...

nba_prizepicks: 0 rows in the table, none in the latest window


nba_novig_team: 0 rows, window 0


nba_pinnacle_team: 0 rows, window 0


nba_prophetx_team: 6 rows, window 6
No points quotes in the latest window. Load the nba_* prop tables, then rerun from this cell.


,book,market_type,rows
0,prophetx,spread,2
1,prophetx,total,2


## Quantiles, copula, count model

Quoted players are matched to silver by normalized name. Home and away are matched to `team_name`. The feature panel is every appearance before the slate date, plus tonight's assumed roster. `starting` is 1 when RotoWire lists that player in the team's expected five, and 0 when the team has a posted game and the player is off the five. A quote whose team has no RotoWire game is skipped until that game is posted. Both models predict only the quoted rows that remain. Knots are sorted so a lower quantile cannot sit above a higher one.

The rate knots are summarized as `rate_hat`, the mean of a 200-point quadrature of the rate sampler, and `rate_cv`, its std / mean. Points given minutes are NB2 with mean `exp(log_kappa) x m^beta x exp(beta2 x log(m/24)^2) x rate_hat` and dispersion `exp(log_alpha) x (m/24)^alpha_minutes x rate_cv^alpha_cv`. Minutes PIT `U` and points PIT `V` are joined by the exported Frank copula. The PMF averages the copula conditional of `V` given `U` over a 50-point grid of minutes quantiles, with mass above 90 in the last bin. The price is the share of that PMF above the line, with a push on whole-number lines. The median is a separate summary.

The copula and count model are read from `copula_spec.json`. The run prints the spec status and flags any bundle or tail file whose hash differs from the one the copula was fit on.

EV is expected profit per $1 at the offered American odds. A side is positive when the simulated win probability clears that side's raw implied probability. `pass` means both sides are inside the juice. `no_vig_over` is the multiplicative no-vig probability. The six draws under the card are one illustration: `U` sets minutes, `V` comes from the copula given `U`, and points are the NB2 quantile at `V`. The PMF is the price.


In [3]:
import math
if quotes.empty:
    print("Skipped. There are no points quotes to price.")
else:
    slate = run_slate(quotes, team_lines)
    card = slate["card"]
    if not slate["unmatched"].empty:
        print("Unmatched quotes")
        display(slate["unmatched"][["book", "player_name", "home_team", "away_team", "reason"]].drop_duplicates())
    if card.empty:
        print("No quoted player matched a team in tonight's games.")
    else:
        if slate["missing"].empty:
            print("Quoted rows have every model feature filled.")
        else:
            print("Model features still empty on the quoted rows")
            display(slate["missing"].style.format({"missing": "{:.0%}"}))
        example = slate["example"]
        quote = example["quote"]
        player = example["player"]
        display(Markdown(
            f"**{quote.player_name}** {quote.line_score} on {quote.book}. "
            f"RotoWire starting flag {int(quote.starting)}. "
            f"Minutes q50 {quote.minutes_q50:.2f}, rate mean {quote.rate_hat:.3f} (cv {quote.rate_cv:.2f}), "
            f"PMF mean {quote.pmf_mean:.2f}, PMF median {quote.pmf_median}."
        ))
        display(example["knots"].round(3))
        display(example["draws"].round(3))
        lo = max(0, int(math.floor(quote.line_score)) - 12)
        hi = min(KMAX, int(math.floor(quote.line_score)) + 12)
        ks = np.arange(lo, hi + 1)
        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.bar(ks, example["pmf"][lo:hi + 1], width=0.8, color="C0")
        ax.axvline(quote.line_score, color="C3", linewidth=1.5)
        ax.set_xlabel("points")
        ax.set_ylabel("probability")
        ax.set_title(f"{quote.player_name} {quote.line_score}  P(over)={quote.sim_over:.3f}")
        plt.show()
        if pd.notna(quote.no_vig_over):
            display(Markdown(
                f"Simulated P(over) {quote.sim_over:.3f} against no-vig {quote.no_vig_over:.3f}. "
                f"Raw over {quote.raw_over:.3f} at {quote.over_american:+.0f}, "
                f"raw under {quote.raw_under:.3f} at {quote.under_american:+.0f}. "
                f"EV over {quote.ev_over:+.3f}, EV under {quote.ev_under:+.3f} per $1. "
                f"Model side: {quote.model_side}."
            ))
        else:
            display(Markdown(
                f"Simulated P(over) {quote.sim_over:.3f}. This quote does not have both American prices, so EV is only filled where a price exists."
            ))
        show = card.drop(columns=["player_id", "game_id", "best_ev"])
        prob_cols = [c for c in show.columns if c.startswith(("sim_", "raw_", "no_vig_", "ev_")) or c in {"minutes_q50", "rate_q50", "rate_hat", "rate_cv", "pmf_mean"}]
        display(show.style.format({column: "{:.3f}" for column in prob_cols}, na_rep=""))


Skipped. There are no points quotes to price.
